# 05 - Feature Relevance Audit

Uses the best tree-based model from notebook 04 to compute permutation importance, then saves only the features that contribute meaningfully to `debug_exports/top_features.json`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from dotenv import load_dotenv
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, mean_squared_error

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
load_dotenv(str(ROOT / ".env"), override=False)

from src.data.fetch_openmeteo import AIR_QUALITY_HOURLY, WEATHER_HOURLY
from src.features.feature_catalog import get_feature_catalog
from src.models.model_configs import MODEL_CONFIGS
from src.models.train import train_model
from src.utils.mongo_client import get_database

sns.set_theme(style="whitegrid")

ARTIFACTS_DIR = ROOT / "debug_exports"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
TREE_MODEL_NAMES = {"lightgbm", "xgboost", "catboost", "random_forest", "extra_trees", "gradient_boosting"}


def load_json(path: Path, default):
    if not path.exists():
        return default
    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except (OSError, json.JSONDecodeError):
        return default


def load_text(path: Path, default: str | None = None) -> str | None:
    if not path.exists():
        return default
    try:
        value = path.read_text(encoding="utf-8").strip()
        return value or default
    except OSError:
        return default


db = get_database()
collection = db["aqi_features_rawalpindi"]
data = pd.DataFrame(list(collection.find()))

if data.empty:
    raise ValueError("aqi_features_rawalpindi is empty")

if "_id" in data.columns:
    data = data.drop(columns=["_id"])

data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

feature_cols = [column for column in get_feature_catalog() if column in data.columns]
# Default window for feature importance (days)
best_window_days = 80
# If you'd like to use a previously saved best window, set use_saved_window = True
use_saved_window = False
if use_saved_window:
    window_payload = load_json(ARTIFACTS_DIR / "best_window_days.json", {})
    if isinstance(window_payload, dict):
        best_window_days = int(window_payload.get("best_window_days", best_window_days))
# Choose a tree model to compute permutation importance. Default to lightgbm if none saved.
best_tree_model_name = load_text(ARTIFACTS_DIR / "best_tree_model_name.txt") or load_text(ARTIFACTS_DIR / "best_model_name.txt")
if not best_tree_model_name:
    best_tree_model_name = "lightgbm"
if best_tree_model_name not in TREE_MODEL_NAMES:
    best_tree_model_name = "lightgbm"

window_start = data["timestamp"].max() - pd.Timedelta(days=best_window_days)
window_frame = data.loc[data["timestamp"] >= window_start].sort_values("timestamp").reset_index(drop=True)

if len(window_frame) <= 72:
    raise ValueError("Not enough rows in the selected window to compute feature importance")


def prepare_training_matrices(window_frame: pd.DataFrame, feature_columns: list[str]):
    horizon = 72
    if len(window_frame) <= horizon:
        return None, None, None

    target = window_frame["european_aqi"].astype(float)
    feature_frame = window_frame[feature_columns].apply(pd.to_numeric, errors="coerce").ffill().fillna(0.0)
    feature_frame = feature_frame.loc[:, feature_frame.nunique(dropna=True) > 1].copy()
    if feature_frame.empty or len(feature_frame) <= horizon:
        return None, None, None

    y = pd.DataFrame([target.iloc[index : index + horizon].values for index in range(len(target) - horizon)])
    x = feature_frame.iloc[: len(y)].copy()
    if x.empty:
        return None, None, None

    return x, y, list(x.columns)


x, y, used_features = prepare_training_matrices(window_frame, feature_cols)
if x is None or y is None:
    raise ValueError("Unable to prepare training matrices for feature importance")

model_config = next((config for config in MODEL_CONFIGS if config.name == best_tree_model_name), None)
if model_config is None:
    raise ValueError(f"Model config not found for {best_tree_model_name}")

model, preds, y_true = train_model(model_config, x, y, horizon=72)
split_idx = int(len(x) * 0.9)
x_val = x.iloc[split_idx:]
y_val = y.iloc[split_idx:]
if x_val.empty or y_val.empty:
    raise ValueError("Validation slice is empty for feature importance")

# Use RMSE scorer wrapper to avoid incompatible sklearn parameter passing
import numpy as _np
from sklearn.metrics import mean_squared_error as _mse

def _rmse(y_true, y_pred):
    return float(_np.sqrt(_mse(y_true, y_pred)))

scorer = make_scorer(_rmse, greater_is_better=False)
importance_result = permutation_importance(model, x_val, y_val, n_repeats=5, random_state=42, scoring=scorer)
importance_df = pd.DataFrame(
    {
        "feature": x.columns,
        "importance": importance_result.importances_mean,
    }
)
importance_df["importance"] = importance_df["importance"].clip(lower=0.0)
importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)

if importance_df["importance"].sum() > 0:
    importance_df["share"] = importance_df["importance"] / importance_df["importance"].sum()
else:
    importance_df["share"] = 0.0
importance_df["cumulative_share"] = importance_df["share"].cumsum()

selected_features = importance_df.loc[importance_df["cumulative_share"] <= 0.95, "feature"].tolist()
if len(selected_features) < 12:
    selected_features = importance_df.head(min(20, len(importance_df)))["feature"].tolist()
selected_features = selected_features[:40]

top_features_payload = {
    "features": selected_features,
    "model_name": best_tree_model_name,
    "best_window_days": best_window_days,
    "feature_count": len(selected_features),
    "selection_rule": "permutation_importance_cumulative_share_0.95",
    "top_importances": importance_df.head(25).to_dict(orient="records"),
}
with open(ARTIFACTS_DIR / "top_features.json", "w", encoding="utf-8") as handle:
    json.dump(top_features_payload, handle, indent=2)

importance_df.to_csv(ARTIFACTS_DIR / "feature_importance_metrics.csv", index=False)

audit = pd.DataFrame(
    {
        "expected_weather_fields": [len(WEATHER_HOURLLY)] if 'WEATHER_HOURLLY' in globals() else [len(WEATHER_HOURLY)],
        "present_weather_fields": [len(set(WEATHER_HOURLY) & set(data.columns))],
        "missing_weather_fields": [len(set(WEATHER_HOURLY) - set(data.columns))],
        "expected_air_fields": [len(AIR_QUALITY_HOURLY)],
        "present_air_fields": [len(set(AIR_QUALITY_HOURLY) & set(data.columns))],
        "missing_air_fields": [len(set(AIR_QUALITY_HOURLY) - set(data.columns))],
        "selected_features": [len(selected_features)],
    }
)

display(audit)
display(importance_df.head(25).rename(columns={"importance": "permutation_importance"}))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
importance_df.head(20).sort_values("importance").plot(kind="barh", x="feature", y="importance", ax=axes[0], color="#2563eb", legend=False)
axes[0].set_title(f"Top Features from {best_tree_model_name}")
axes[0].set_xlabel("Permutation importance")
axes[0].set_ylabel("Feature")

missingness = data.isna().mean().sort_values(ascending=False).head(20)
missingness.plot(kind="barh", ax=axes[1], color="#dc2626")
axes[1].set_title("Top Missing Columns")
axes[1].set_xlabel("Missing fraction")

plt.tight_layout()

print(f"Saved {ARTIFACTS_DIR / 'top_features.json'}")
print(f"Selected {len(selected_features)} features from {best_tree_model_name} on the {best_window_days}-day window")